In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install datasets transformers accelerate bitsandbytes sentencepiece pandas matplotlib tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.1 MB/s eta 0:00:00


In [ ]:
import os
import random
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig

In [ ]:
MODEL_NAME = "bigscience/mt0-xl"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)
print("Model parameter device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/797 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model parameter device: cuda:0


In [ ]:
def load_and_split(lang_code: str, seed: int):
    ds = load_dataset("facebook/belebele", lang_code, split="test")
    ds = ds.shuffle(seed=seed)
    exemplar_pool = ds.select(range(30))
    remaining = ds.select(range(30, 30 + 150))
    return exemplar_pool, remaining

In [ ]:
LETTERS = ["A", "B", "C", "D"]

def format_question_block(row, show_answer: bool):
    options = [row["mc_answer1"], row["mc_answer2"], row["mc_answer3"], row["mc_answer4"]]
    block = f"Passage: {row['flores_passage']}\n"
    block += f"Question: {row['question']}\n"
    for letter, opt in zip(LETTERS, options):
        block += f"{letter}. {opt}\n"
    if show_answer:
        correct_letter = LETTERS[int(row["correct_answer_num"]) - 1]
        block += f"Answer: {correct_letter}\n"
    else:
        block += "Answer:"
    return block

def build_prompt(exemplars, target_row):
    parts = [format_question_block(ex, show_answer=True) for ex in exemplars]
    parts.append(format_question_block(target_row, show_answer=False))
    instructions = (
        "Answer each multiple-choice reading comprehension question "
        "using only the passage provided. Respond with a single "
        "letter (A, B, C, or D) and nothing else."
    )
    return instructions + "\n\n" + "\n\n".join(parts)

In [ ]:
def sample_exemplars(exemplar_pool, k, seed):
    rng = random.Random(seed)
    if k == 0:
        return []
    indices = rng.sample(range(len(exemplar_pool)), k)
    return [exemplar_pool[i] for i in indices]

In [ ]:
tokenizer.truncation_side = "left"

def call_llm(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=5120).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=5, do_sample=False)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

def parse_prediction(raw_output: str):
    for letter in LETTERS:
        if letter in raw_output.upper():
            return letter
    return None

In [ ]:
exemplar_pool, test_pool = load_and_split("sin_Sinh", 42)
exemplars = sample_exemplars(exemplar_pool, 8, 42)
prompt = build_prompt(exemplars, test_pool[0])
start = time.time()
_ = call_llm(prompt)
elapsed = time.time() - start
print(f"{elapsed:.2f}s per call")
print(f"Estimated total (900 calls): {elapsed * 900 / 60:.1f} minutes")

README.md:   0%|          | 0.00/25.5k [00:00<?, ?B/s]

sin_Sinh.jsonl:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

4.35s per call
Estimated total (900 calls): 65.3 minutes


In [ ]:
CONFIG_SCALE = {
    "languages": {"English": "eng_Latn", "Sinhala": "sin_Sinh"},
    "n_seeds": 3,
    "random_seed_base": 42,
    "output_dir": "/content/drive/MyDrive/belebele_results/results_scaled",
}
os.makedirs(CONFIG_SCALE["output_dir"], exist_ok=True)

scale_results = []
for lang_name, lang_code in CONFIG_SCALE["languages"].items():
    exemplar_pool, test_pool = load_and_split(lang_code, CONFIG_SCALE["random_seed_base"])

    for seed_offset in range(CONFIG_SCALE["n_seeds"]):
        seed = CONFIG_SCALE["random_seed_base"] + seed_offset
        exemplars = sample_exemplars(exemplar_pool, 8, seed)
        correct, total = 0, 0
        letter_counts = {"A": 0, "B": 0, "C": 0, "D": 0, "None": 0}

        for row in tqdm(test_pool, desc=f"{lang_name} | seed={seed} | SCALED"):
            prompt = build_prompt(exemplars, row)
            try:
                raw = call_llm(prompt)
            except Exception as e:
                print(f"Error: {e}")
                torch.cuda.empty_cache()
                continue
            pred = parse_prediction(raw)
            gold = LETTERS[int(row["correct_answer_num"]) - 1]
            total += 1
            if pred == gold:
                correct += 1
            letter_counts[pred if pred else "None"] += 1

        torch.cuda.empty_cache()
        scale_results.append({
            "language": lang_name, "seed": seed,
            "accuracy": correct / total if total else 0,
            "n_scored": total, **letter_counts,
        })
        pd.DataFrame(scale_results).to_csv(f"{CONFIG_SCALE['output_dir']}/scaled_k8_results.csv", index=False)

scale_df = pd.DataFrame(scale_results)
print(scale_df)
print("\nAggregated:")
print(scale_df.groupby("language")["accuracy"].agg(["mean", "std"]))
print("\nC-selection rate:")
for lang, group in scale_df.groupby("language"):
    print(f"{lang}: C selected in {(group['C'].sum() / group['n_scored'].sum()):.1%} of predictions")

eng_Latn.jsonl:   0%|          | 0.00/836k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/900 [00:00<?, ? examples/s]

English | seed=42 | SCALED: 100%|██████████| 150/150 [04:08<00:00,  1.66s/it]
English | seed=43 | SCALED: 100%|██████████| 150/150 [04:29<00:00,  1.80s/it]
English | seed=44 | SCALED: 100%|██████████| 150/150 [04:12<00:00,  1.68s/it]
Sinhala | seed=42 | SCALED: 100%|██████████| 150/150 [08:30<00:00,  3.41s/it]
Sinhala | seed=43 | SCALED: 100%|██████████| 150/150 [09:21<00:00,  3.74s/it]
Sinhala | seed=44 | SCALED: 100%|██████████| 150/150 [08:06<00:00,  3.25s/it]

  language  seed  accuracy  n_scored   A   B    C   D  None
0  English    42  0.640000       150  17  16   89  28     0
1  English    43  0.473333       150  10  14  109  17     0
2  English    44  0.533333       150  13  22  101  14     0
3  Sinhala    42  0.460000       150  11  19  100  20     0
4  Sinhala    43  0.493333       150  23  20  100   7     0
5  Sinhala    44  0.520000       150  26  23   84  17     0

Aggregated:
              mean       std
language                    
English   0.548889  0.084415
Sinhala   0.491111  0.030062

C-selection rate:
English: C selected in 66.4% of predictions
Sinhala: C selected in 63.1% of predictions


In [ ]:
print(scale_df[["language", "seed", "accuracy", "n_scored"]])

  language  seed  accuracy  n_scored
0  English    42  0.640000       150
1  English    43  0.473333       150
2  English    44  0.533333       150
3  Sinhala    42  0.460000       150
4  Sinhala    43  0.493333       150
5  Sinhala    44  0.520000       150


In [ ]:
for lang, group in scale_df.groupby("language"):
    total_n = group["n_scored"].sum()
    c_total = group["C"].sum()
    print(f"{lang}: C selected in {c_total}/{total_n} = {c_total/total_n:.1%} of predictions (n=3 seeds)")

English: C selected in 299/450 = 66.4% of predictions (n=3 seeds)
Sinhala: C selected in 284/450 = 63.1% of predictions (n=3 seeds)


In [ ]:
from scipy.stats import binomtest
# English: 299/450 for C
print(binomtest(299, 450, p=0.25, alternative='greater'))
# Sinhala: 284/450
print(binomtest(284, 450, p=0.25, alternative='greater'))

BinomTestResult(k=299, n=450, alternative='greater', statistic=0.6644444444444444, pvalue=3.106267232986865e-76)
BinomTestResult(k=284, n=450, alternative='greater', statistic=0.6311111111111111, pvalue=4.211720018800741e-65)
